In [ ]:
import pandas as pd 
import numpy as np 
import plotly.express as px 
import plotly.graph_objects as go 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error,r2_score,confusion_matrix,classification_report, roc_curve,roc_auc_score

In [ ]:
df = pd.read_csv(r"C:\Users\chett\Downloads\diabetes.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Outcome'].value_counts()

In [ ]:
px.histogram(
    df,
    x='Glucose',
    nbins=30,
    title= 'Distribution of Glucose Level',
    color_discrete_sequence=["orange"]
)
    

In [ ]:
px.scatter(
    df,
    x="Age",
    y="Glucose",
    color="Outcome",
    title= "Age vs Glucose colored by Diabetes Outcome"
)

In [ ]:
(df == 0).sum()

In [ ]:
cols_with_zero=[
    'Glucose',
    'BloodPressure',
    'BMI',
    'Insulin',
    'SkinThickness'
]

In [ ]:
df[cols_with_zero]=df[cols_with_zero].replace(0,np.nan)

In [ ]:
df

In [ ]:
df[cols_with_zero]=df[cols_with_zero].fillna(df[cols_with_zero].median())

In [ ]:
df

In [ ]:
X_lr=df.drop(['Glucose','Outcome'], axis=1)
Y_lr=df['Glucose']
             

In [ ]:
X_lr_train,X_lr_test,Y_lr_train,Y_lr_test = train_test_split(
    X_lr,Y_lr, train_size = 0.2 , random_state = 42 
)

In [ ]:
scaler = StandardScaler()
X_lr_train_scaled = scaler.fit_transform(X_lr_train)
X_lr_test_scaled = scaler.transform(X_lr_test)

In [ ]:
lr_model= LinearRegression()

In [ ]:
lr_model.fit(X_lr_train , Y_lr_train)

In [ ]:
y_lr_pred = lr_model.predict(X_lr_test_scaled)

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=Y_lr_test,
    mode='markers',
    name='Actual Glucose'
))

fig.add_trace(go.Scatter(
    y=y_lr_pred,
    mode='markers',
    name='Predicted Glucose'
    
))

fig.update_layout(
    title="Actual vs Predicted Glucose Levels",
    xaxis_title="Sample Index",
    yaxis_title="Glucose Level"
)

fig.show()

In [ ]:
rmse = np.sqrt(mean_squared_error(Y_lr_test, y_lr_pred))
r2 = r2_score(Y_lr_test, y_lr_pred)

rmse, r2 

In [ ]:
X = df.drop('Outcome', axis = 1)
Y = df['Outcome']

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(
    X,Y, test_size = 0.2 , random_state = 42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_model = LogisticRegression(max_iter=1000)

In [ ]:
log_model.fit(X_train_scaled,Y_train)

In [ ]:
y_pred = log_model.predict(X_test_scaled)
y_pred

In [ ]:
y_prob = log_model.predict_proba(X_test_scaled)[:,1]
y_prob

In [ ]:
prob_df = pd.DataFrame({
    "Patient_Index": range(len(y_prob)),
    "Diabetes_Risk_Probability": y_prob,
    "Predicted_Class": y_pred
})

px.bar(
    prob_df,
    x="Patient_Index",
    y="Diabetes_Risk_Probability",
    color="Predicted_Class",
    title="Diabetes Risk Probability per Patient",
    labels={"Diabetes_Risk_Probability": "Probability of Diabetes"}
)


In [ ]:
cm = confusion_matrix(Y_test , y_pred)
cm

In [ ]:
cm_df = np.array(cm)

px.imshow(
    cm_df,
    text_auto=True,
    labels=dict(x="Predicted Label", y="Actual Label"),
    x=["No Diabetes", "Diabetes"],
    y=["No Diabetes", "Diabetes"],
    title="Confusion Matrix for Diabetes Prediction"
)

In [ ]:
print(classification_report(Y_test,y_pred))

In [ ]:
fpr,tpr,thresholds = roc_curve(Y_test, y_prob)
auc_score = roc_auc_score(Y_test,y_prob)

auc_score

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines+markers',
    name=f'Logistic Regression (AUC = {auc_score:.2f})'
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(dash='dash')
))

fig.update_layout(
    title='ROC Curve for Diabetes Prediction',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate'
)

fig.show()
